In [13]:
import os
import ast
import json
import requests
import pandas as pd

# ETL (Extraer, Transformar, Cargar) 

In [15]:
url = f"https://rickandmortyapi.com/api/character/1"
response = requests.get(url)
print(response)

<Response [200]>


In [16]:
datos = response.json()
datos

{'id': 1,
 'name': 'Rick Sanchez',
 'status': 'Alive',
 'species': 'Human',
 'type': '',
 'gender': 'Male',
 'origin': {'name': 'Earth (C-137)',
  'url': 'https://rickandmortyapi.com/api/location/1'},
 'location': {'name': 'Citadel of Ricks',
  'url': 'https://rickandmortyapi.com/api/location/3'},
 'image': 'https://rickandmortyapi.com/api/character/avatar/1.jpeg',
 'episode': ['https://rickandmortyapi.com/api/episode/1',
  'https://rickandmortyapi.com/api/episode/2',
  'https://rickandmortyapi.com/api/episode/3',
  'https://rickandmortyapi.com/api/episode/4',
  'https://rickandmortyapi.com/api/episode/5',
  'https://rickandmortyapi.com/api/episode/6',
  'https://rickandmortyapi.com/api/episode/7',
  'https://rickandmortyapi.com/api/episode/8',
  'https://rickandmortyapi.com/api/episode/9',
  'https://rickandmortyapi.com/api/episode/10',
  'https://rickandmortyapi.com/api/episode/11',
  'https://rickandmortyapi.com/api/episode/12',
  'https://rickandmortyapi.com/api/episode/13',
  'htt

In [18]:
list_data = []
for i in range(5):
  url = f"https://rickandmortyapi.com/api/character/{i+1}"
  response = requests.get(url).json()
  data_to_save = {
      'id':response['id'],
      'name':response['name'],
      'gender':response['gender'],
      'type': response['type'],
      'species': response['species'],
      'count_episodes':len(response['episode'])
  }
  list_data.append(data_to_save)
  print(data_to_save)

#pd.DataFrame(list_data).to_parquet('data_api.parquet', engine= 'pyarrow', index=False)
pd.DataFrame(list_data).to_parquet('data_api.parquet', engine='fastparquet', index=False)

{'id': 1, 'name': 'Rick Sanchez', 'gender': 'Male', 'type': '', 'species': 'Human', 'count_episodes': 51}
{'id': 2, 'name': 'Morty Smith', 'gender': 'Male', 'type': '', 'species': 'Human', 'count_episodes': 51}
{'id': 3, 'name': 'Summer Smith', 'gender': 'Female', 'type': '', 'species': 'Human', 'count_episodes': 42}
{'id': 4, 'name': 'Beth Smith', 'gender': 'Female', 'type': '', 'species': 'Human', 'count_episodes': 42}
{'id': 5, 'name': 'Jerry Smith', 'gender': 'Male', 'type': '', 'species': 'Human', 'count_episodes': 39}


In [24]:
data_parquet_etl = pd.read_parquet('data_api.parquet', engine= 'fastparquet')
data_parquet_etl.head()

,id,name,gender,type,species,count_episodes
0,1,Rick Sanchez,Male,,Human,51
1,2,Morty Smith,Male,,Human,51
2,3,Summer Smith,Female,,Human,42
3,4,Beth Smith,Female,,Human,42
4,5,Jerry Smith,Male,,Human,39


In [25]:
data_parquet_etl.shape

(5, 6)

# ELT (Extraer, Cargar, Transformar) son dos métodos para mover y preparar datos desde varios sitios de origen hasta un sistema central de almacenamiento.

In [26]:
path = "api_rickmorty/"
if not os.path.exists(path):
    os.makedirs(path)
for i in range(5):
  url = f"https://rickandmortyapi.com/api/character/{i+1}"
  response = requests.get(url).json()
  with open(f"{path}{i+1}.json", "w") as file:
    file.write(str(response))

In [28]:
path = "api_rickmorty/"
list_data = []
for i in range(5):
  with open(f"{path}{i+1}.json", "r") as file:
    json_file = ast.literal_eval(file.read())
  data_to_save = {
      'id':json_file['id'],
      'name':json_file['name'],
      'gender':json_file['gender'],
      'count_episodes':len(json_file['episode']),
      'status':json_file['status']
  }
  list_data.append(data_to_save)
pd.DataFrame(list_data).to_parquet('data_api_elt.parquet',engine='fastparquet', index=False)